<a href="https://colab.research.google.com/github/rakeshma-72/LLM/blob/main/EXPERIMENT_4_Attention_Mechanism_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers bertviz torch --quiet

In [9]:
import torch
from transformers import AutoTokenizer, AutoModel
from bertviz import head_view, model_view

# Check whether a GPU is available; attention extraction runs fine on CPU
# too, but GPU speeds up the forward pass for larger models.

In [10]:
# Install the required packages (run once in Jupyter/Colab)
!pip install transformers bertviz torch --quiet

# Core imports
import torch
from transformers import AutoTokenizer, AutoModel
from bertviz import head_view, model_view

# Check whether a GPU is available; attention extraction runs fine on CPU
# too, but GPU speeds up the forward pass for larger models.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [11]:
# Name of the pretrained checkpoint to load from the Hugging Face Hub
MODEL_NAME = "bert-base-uncased"

# Load the tokenizer that matches the model's vocabulary
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the model itself, requesting attention weights in the output and
# moving the model to the selected device (GPU if available)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    output_attentions=True  # <-- key flag: returns attention tensors
).to(device)

# Put the model in evaluation mode (disables dropout, etc.)
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [12]:
# Define the input text to analyze
sentence = "The cat sat on the mat because it was tired."

# Tokenize the sentence. Calling the tokenizer directly handles special tokens
# and all configurations automatically, replacing the older 'encode_plus'.
inputs = tokenizer(sentence, return_tensors="pt").to(device)

# Run a forward pass without computing gradients (inference only)
with torch.no_grad():
    outputs = model(**inputs)

# outputs.attentions -> tuple of length num_layers
# each element shape: (batch_size, num_heads, seq_len, seq_len)
attention = outputs.attentions
print("Number of layers:", len(attention))
print("Shape of one layer's attention tensor:", attention[0].shape)

# Convert token IDs back to readable tokens for visualization/analysis
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("Tokens:", tokens)

Number of layers: 12
Shape of one layer's attention tensor: torch.Size([1, 12, 13, 13])
Tokens: ['[CLS]', 'the', 'cat', 'sat', 'on', 'the', 'mat', 'because', 'it', 'was', 'tired', '.', '[SEP]']


In [13]:
# Interactive view of every head in a single layer (arcs between tokens)
head_view(attention, tokens)

<IPython.core.display.Javascript object>

In [14]:
# Compact grid view: all layers x all heads in one screen
model_view(attention, tokens)


<IPython.core.display.Javascript object>

In [15]:
import numpy as np

LAYER = 5  # choose a layer index (0 to num_layers-1)
QUERY_TOKEN_IDX = tokens.index("it")  # analyze the pronoun "it"

# attention[LAYER] shape: (1, num_heads, seq_len, seq_len)
layer_attn = attention[LAYER][0]   # drop batch dim -> (heads, seq, seq)
avg_attn = layer_attn.mean(dim=0)  # average across heads -> (seq, seq)

# Attention distribution FROM the query token TO every other token
scores = avg_attn[QUERY_TOKEN_IDX].cpu().numpy()

# Sort tokens by how much attention the query token pays to them
ranked = sorted(zip(tokens, scores), key=lambda x: x[1], reverse=True)

print(f"Top tokens attended to by '{tokens[QUERY_TOKEN_IDX]}' "
      f"(layer {LAYER}, averaged over heads):")
for tok, score in ranked[:5]:
    print(f"  {tok:<12s}-> {score:.4f}")

Top tokens attended to by 'it' (layer 5, averaged over heads):
  [SEP]       -> 0.4985
  tired       -> 0.1004
  because     -> 0.0934
  was         -> 0.0730
  [CLS]       -> 0.0498


In [16]:
# Simple automated interpretation helper
def interpret(tokens, scores, top_k=3):
    ranked = sorted(zip(tokens, scores), key=lambda x: x[1], reverse=True)
    top = [t for t, s in ranked[:top_k] if t not in ("[CLS]", "[SEP]")]
    return top

top_context_words = interpret(tokens, scores)
print("The model resolves the pronoun mainly using:", top_context_words)

# Expected qualitative result: "cat" may outrank "mat", suggesting the model
# grounds "it" using semantic context rather than mere positional proximity.
# This is not guaranteed: results vary by layer, and averaged attention often
# favors [CLS], [SEP], or punctuation.

The model resolves the pronoun mainly using: ['tired', 'because']
